# PPO
不多说，直接开撕


In [ ]:
# 每日一导
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

In [ ]:
from dataclasses import dataclass

@dataclass
class PPOConfig:
    obs_dim: int
    act_dim: int

    hidden_dim: int = 128
    lr: float = 3e-4

    gamma: float = 0.99
    lam: float = 0.95

    clip_ratio: float = 0.2
    vf_coef: float = 0.5
    ent_coef: float = 0.01

    train_epochs: int = 10
    minibatch_size: int = 64
    max_grad_norm: float = 0.5

    device: str = "cpu"

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self,obs_dim,act_dim,hidden_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim,hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.Tanh(),
        )
        self.actor_head = nn.Linear(hidden_dim,act_dim)
        self.critic_head = nn.Linear(hidden_dim,1)
    def forward(self,obs:torch.Tensor):
        """
        obs: [N, obs_dim]
        return:
            logits: [N, act_dim]
            value:  [N]
        """
        feat = self.shared(obs)
        logits = self.actor_head(feat)
        value = self.critic_head(feat).squeeze(-1)
        return logits,value
    # 工具函数，用于得到分布和value
    def get_distri_and_value(self,obs):
        logits,value = self.forward(obs)
        distri = Categorical(logits=logits)
        return distri,value
    
    # 写act采样
    @torch.no_grad()
    def act(self,obs):
        """
        obs: [B, obs_dim]
        return:
            action: [B]
            log_prob: [B]
            value: [B]
        """
        distri,value = self.get_distri_and_value(obs)
        action = distri.sample()
        log_prob = distri.log_prob(action)
        return action,log_prob,value
    
    # update用的
    def evaluate_actions(self,obs,actions):
        """
        obs: [B, obs_dim]
        return:
            action: [B]
            log_prob: [B]
            value: [B]
        """
        distri,value = self.get_distri_and_value(obs)
        new_log_prob = distri.log_prob(actions)
        entropy = distri.entropy()
        return new_log_prob,entropy,value
    
    

In [ ]:

class RolloutBuffer:
    def __init__(self):
        self.clear()
    def clear(self):
        self.obs = []
        self.actions = []
        self.log_probs=[]
        self.rewards = []
        self.dones = []
        self.values= []
    def add(self,obs,action,log_prob,reward,done,value):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.dones.append(done)
        self.values.append(value)
    
    def get_tensors(self, device="cpu"):
        data = {
            "obs": torch.stack(self.obs).to(device),             # [T, B, obs_dim]
            "actions": torch.stack(self.actions).to(device),     # [T, B]
            "log_probs": torch.stack(self.log_probs).to(device), # [T, B]
            "rewards": torch.stack(self.rewards).to(device),     # [T, B]
            "dones": torch.stack(self.dones).to(device),         # [T, B]
            "values": torch.stack(self.values).to(device),       # [T, B]
        }
        return data

In [ ]:
class PPO:
    def __init__(self,cfg:PPOConfig):
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.policy=ActorCritic(
            obs_dim=cfg.obs_dim,
            act_dim=cfg.act_dim,
            hidden_dim =cfg.hidden_dim,
        ).to(self.device)

        self.optimizer = torch.optim.Adam(self.policy.parameters(),lr=cfg.lr)
        self.buffer = RolloutBuffer() 
    @torch.no_grad()
    def select_action(self,obs):
        """
        obs: [B, obs_dim]
        return:
            action: [B]
            log_prob: [B]
            value: [B]
        """
        obs = obs.to(self.device)
        action,log_prob,value=self.policy.act(obs)
        return action,log_prob,value
    
    def store_transition(self,obs,action,log_prob,reward,done,value):
        self.buffer.add(obs,action,log_prob,reward,done,value)
    
    def compute_gae_and_returns(
            self,
            rewards,
            dones,
            values,
            last_value,
    ):
        """
        rewards: [T, B]
        dones:   [T, B]
        values:  [T, B]
        last_value: [B]

        return:
            advantages: [T, B]
            returns: [T, B]
        """
        T,B=rewards.shape
        advantages = torch.zeros_like(rewards,device=self.device)
        gae = torch.zeros(B,device=self.device)

        for t in reversed(range(T)):
            if t == T-1:
                next_value = last_value
            else:
                next_value=values[t+1]
            not_done = 1.0 - dones(t)
            
            delta = rewards[t]+self.cfg.gamma *next_value*not_done-values[t]
            gae = delta + self.gamma*self.cfg.lam*not_done*gae
            advantages[t] = gae
        
        returns =advantages + values
        return advantages,returns